### import + path

In [7]:
import pandas as pd
from pathlib import Path
import sqlite3

### convert to sqlite

In [8]:
##create street_crimes

#paths/connection
path = Path("../data")
db_name = "../data/police_data.db"  
db_connection = sqlite3.connect(db_name)

#files end in 3 ways: *-street, *-stop-and-search, *-outcomes, edit to work with new
orig_files = list(path.rglob("*-street.csv"))
print(len(orig_files))

#renaming/defining which ones to keep
#dropped: lsoa name, location, reported by, falls within
column_mapping = {
    'Crime ID': 'crime_id',
    'Month': 'month',
    'Longitude': 'longitude',
    'Latitude': 'latitude',
    'LSOA code': 'lsoa_code',
    'Crime type': 'crime_type',
    'Last outcome category': 'last_outcome'
}

#start count to see progress
successful = 0

for i, file in enumerate(orig_files):
    try:
        #takes only columns mentioned before, low_memory bc pandas warnings
        df = pd.read_csv(file, usecols=column_mapping.keys(), low_memory=False)
            
        #rename the columns as above
        df = df.rename(columns=column_mapping)
            
        #write to db
        df.to_sql("street_crimes", db_connection, if_exists="append", index=False)
        successful += 1
        
        if (i + 1) % 50 == 0:
            print(f"processed {i + 1} files")
                
    except Exception as e:
        print(f"not processed: {file.name}: {e}")



#index for based on lsoa
db_connection.execute("CREATE INDEX IF NOT EXISTS idx_street_lsoa ON street_crimes(lsoa_code);")

#index for crimes by date
db_connection.execute("CREATE INDEX IF NOT EXISTS idx_street_month ON street_crimes(month);")


db_connection.close()
print(f"{successful} files into {db_name}")

8107
processed 50 files
processed 100 files
processed 150 files
processed 200 files
processed 250 files
processed 300 files
processed 350 files
processed 400 files
processed 450 files
processed 500 files
processed 550 files
processed 600 files
processed 650 files
processed 700 files
processed 750 files
processed 800 files
processed 850 files
processed 900 files
processed 950 files
processed 1000 files
processed 1050 files
processed 1100 files
processed 1150 files
processed 1200 files
processed 1250 files
processed 1300 files
processed 1350 files
processed 1400 files
processed 1450 files
processed 1500 files
processed 1550 files
processed 1600 files
processed 1650 files
processed 1700 files
processed 1750 files
processed 1800 files
processed 1850 files
processed 1900 files
processed 1950 files
processed 2000 files
processed 2050 files
processed 2100 files
processed 2150 files
processed 2200 files
processed 2250 files
processed 2300 files
processed 2350 files
processed 2400 files
process

In [9]:
##create lsoa_demographic

file_path = "../data/2025_all_iod_scores_ranks_deciles.csv" 
db_connection = sqlite3.connect("../data/police_data.db")

#define new column for easiness later
column_mapping = {
    'LSOA code': 'lsoa_code',
    'Income Score': 'income_score',
    'Employment Score': 'employment_score',
    'Education Skills and Training Score': 'education_score',
    'Health Deprivation and Disability Score': 'health_score',
    'Barriers to Housing and Services Score': 'barrier_score',
    'Living Environment Score': 'living_score',
    'Total population': 'pop',
    'Dependent Children': 'child_pop',
    'Older population': 'old_pop',
    'Working age population': 'working_pop'
}

try:
    #load only columns mentioned in column_mapping
    df = pd.read_csv(file_path, usecols=column_mapping.keys())
    
    #rename the columns
    df = df.rename(columns=column_mapping)
    
    #write to db
    df.to_sql("lsoa_demographics", db_connection, if_exists="replace", index=False)
    print(len(df))
    
    #create index for faster querying
    db_connection.execute("CREATE INDEX IF NOT EXISTS idx_lsoa_code ON lsoa_demographics(lsoa_code);")

#just in case
except Exception as e:
    print(f"not read: {e}")


db_connection.close()

33755


In [10]:
##create lsoa_info

file_path = "../data/2025_all_iod_scores_ranks_deciles.csv" 
db_connection = sqlite3.connect("../data/police_data.db")

#define new column for easiness later
column_mapping = {
    'LSOA code': 'lsoa_code',
    'LSOA name': 'lsoa_name',
    'Local Authority District code': 'loc_auth_code',
    'Local Authority District name': 'loc_auth_name'
}

try:
    #load only columns mentioned in column_mapping
    df = pd.read_csv(file_path, usecols=column_mapping.keys())
    
    #rename the columns
    df = df.rename(columns=column_mapping)
    
    #write to db
    df.to_sql("lsoa_info", db_connection, if_exists="replace", index=False)
    print(len(df))
    
    #create index for faster querying
    db_connection.execute("CREATE INDEX IF NOT EXISTS idx_lsoa_code ON lsoa_info(lsoa_code);")

#just in case
except Exception as e:
    print(f"not read: {e}")


db_connection.close()
print(f"{successful} files into {db_name}")

33755
8107 files into ../data/police_data.db


In [ ]:
#creating pfa_weather

#data is formatted differently so need to reshape it and match with pfa codes from lsoa_info

db_name = "../data/police_data.db"
file_path = "../data/weather_dataset.csv" 
db_connection = sqlite3.connect(db_name)

#cleaning function to standardize police force names for matching
def clean_pfa_name(series):
    return (series.astype(str)
            .str.lower()
            .str.replace(' constabulary', '', regex=False)
            .str.replace(' police', '', regex=False)
            .str.replace(' service', '', regex=False)
            .str.replace('&', 'and', regex=False)
            .str.strip())

#try to load the weather data and process it
try:
    print("Loading weather data...")
    df = pd.read_csv(file_path)
    
    #manual fix
    manual_fixes = {
        'City of London Police': 'London, City of'
    }
    df['policeForce'] = df['policeForce'].replace(manual_fixes)
    
    #reshape it using melt and pivot_table to get it into long format with columns for each measurement
    print("Reshaping weather data from Wide to Long format...")
    melted = df.melt(id_vars=['policeForce', 'weatherStation'], var_name='time_var', value_name='val')
    melted[['year', 'month_num', 'measurement']] = melted['time_var'].str.split('_', expand=True)
    melted['month'] = melted['year'] + '-' + melted['month_num'].str.zfill(2)
    weather_long = melted.pivot_table(index=['policeForce', 'month'], columns='measurement', values='val').reset_index()
    weather_long.columns.name = None 
    
    #map it with the PFA codes from lsoa_info, using string cleaning to match names
    print("Fetching PFA mapping from lsoa_info...")
    mapping_query = """
    SELECT DISTINCT pfa_name, pfa_code 
    FROM lsoa_info 
    WHERE pfa_name IS NOT NULL;
    """
    pfa_mapping = pd.read_sql(mapping_query, db_connection)
    
    weather_long['match_name'] = clean_pfa_name(weather_long['policeForce'])
    pfa_mapping['match_name'] = clean_pfa_name(pfa_mapping['pfa_name'])
    
    #left match to keep all weather data, even if some forces don't match
    weather_long = weather_long.merge(pfa_mapping, on='match_name', how='left')
    
    #check for missing codes again
    missing_codes = weather_long[weather_long['pfa_code'].isna()]['policeForce'].unique()
    
    #report only errors about English police
    expected_missing = ['Dyfed-Powys Police', 'Gwent Police', 'North Wales Police', 'South Wales Police', 'Police Service of Northern Ireland']
    actual_errors = [m for m in missing_codes if m not in expected_missing]
    
    if len(actual_errors) > 0:
        print(f"\n[!] Still failing to match these specific forces: {actual_errors}")
    else:
        print("\nPerfect Match! All active English Police Forces successfully linked. (Wales and NI safely ignored).")
        
    #clean up the extra columns
    weather_long = weather_long.drop(columns=['policeForce', 'match_name'])
    
    #drop unmapped regions
    weather_long = weather_long.dropna(subset=['pfa_code'])
    
    #write to db
    print("writing final to db")
    weather_long.to_sql("pfa_weather", db_connection, if_exists="replace", index=False)
    
    db_connection.execute("CREATE INDEX IF NOT EXISTS idx_weather_pfa ON pfa_weather(pfa_name, month);")
    db_connection.execute("CREATE INDEX IF NOT EXISTS idx_weather_pfacode ON pfa_weather(pfa_code, month);")
    
    print(f"{len(weather_long)} rows written")

except Exception as e:
    print(f"error: {e}")

finally:
    db_connection.close()

Loading weather data...
Reshaping weather data from Wide to Long format...
Fetching PFA mapping from lsoa_info...

Perfect Match! All active English Police Forces successfully linked. (Wales and NI safely ignored).
Writing clean table to database...
Success! 7644 rows written to 'pfa_weather' in ../data/police_data.db.
